<a href="https://colab.research.google.com/github/susanavenda/data_cambridge/blob/main/Venda_Susana_CAM_C101_W6_Mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-project 5.3: Detecting anomalous activity in a ship's engine

**A report on predictive maintenance using anomaly detection.**


# Introduction

This report addresses the problem of **predictive maintenance** for a shipping fleet. Time-based maintenance leads to unnecessary cost and to catastrophic engine failure at sea. The goal is to detect anomalous engine behaviour from six sensor features (engine RPM, lubrication oil pressure and temperature, fuel pressure, coolant pressure and temperature) so that interventions can be made before failure.

We use a real dataset , expecting anomalies to represent **1–5%** of data. We combine **statistical** (IQR) and **machine learning** (One-Class SVM, Isolation Forest) methods, with 2D PCA for visualisation.

Feature engineering extends the six raw sensor channels into twelve features to improve model sensitivity.

Logical flow:
  Raw data → EDA → Scaling → Baseline IQR → Tuned OCSVM → Tuned IF
  → PCA comparison → Engineered features → Consensus → Recommendations



#1. Data



##1.1 Setup and Data Loading



*   Import necessary Python libraries (pandas, numpy, matplotlib, seaborn, sklearn).
*   Load the dataset from the GitHub repository.
*   Create functions to be reused.







In [ ]:
# @title Setup — Imports, Design System & Constants
import sys
import subprocess
import matplotlib.pyplot as plt

try:
    import plotly
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from IPython.display import display, HTML

try:
    from ydata_profiling import ProfileReport
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ydata-profiling"])
    from ydata_profiling import ProfileReport

try:
    import google.colab  # noqa
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "notebook"

pio.templates.default = "plotly_dark"

C_NORMAL  = "#38BDF8"
C_ANOMALY = "#F87171"
C_ACCENT  = "#A78BFA"
BG_CARD   = "#0F172A"
BG_PLOT   = "#1E293B"

PLOTLY_LAYOUT = dict(
    paper_bgcolor = BG_CARD,
    plot_bgcolor  = BG_PLOT,
    font          = dict(family="Inter, system-ui, sans-serif", color="#CBD5E1", size=12),
    title_font    = dict(size=16, color="#F1F5F9", family="Inter, system-ui, sans-serif"),
    legend        = dict(bgcolor="rgba(0,0,0,0)", bordercolor="#334155", borderwidth=1),
    margin        = dict(l=50, r=30, t=60, b=50),
    xaxis         = dict(gridcolor="#1E293B", linecolor="#334155", zerolinecolor="#334155"),
    yaxis         = dict(gridcolor="#1E293B", linecolor="#334155", zerolinecolor="#334155"),
)

np.random.seed(42)

DATA_URL      = "https://raw.githubusercontent.com/fourthrevlxd/cam_dsb/main/engine.csv"
RANDOM_STATE  = 42
TARGET_NU     = 0.03
CONTOUR_RES   = 300
NU_VALUES     = [0.01, 0.03, 0.05]
GAMMA_VALUES  = [0.1, 0.5, 0.7]
CONTAM_LEVELS = [0.01, 0.03, 0.05]
RAW_SENSORS   = [
    'Engine rpm', 'Lub oil pressure', 'Fuel pressure',
    'Coolant pressure', 'lub oil temp', 'Coolant temp'
]
ENGINEERED_FEATURES = [
    'lub_efficiency', 'cooling_delta', 'power_intensity',
    'rpm_squared', 'log_fuel_pressure', 'sqrt_rpm'
]

_table_counter  = 0
_figure_counter = 0




In [ ]:
# @title  Setup — Display Helper Functions

def _next_table() -> int:
    global _table_counter
    _table_counter += 1
    return _table_counter

def _next_figure() -> int:
    global _figure_counter
    _figure_counter += 1
    return _figure_counter

def display_table(df: pd.DataFrame, title: str = "") -> None:
    n           = _next_table()
    full_title  = f"Table {n}" + (f", {title}" if title else "")
    header_html = (
        f"<p style='font-family:Inter,sans-serif;font-size:11px;font-weight:600;"
        f"color:#64748B;letter-spacing:.08em;text-transform:uppercase;"
        f"margin:24px 0 6px'>{full_title}</p>"
    )
    styled = (
        df.style
        .set_table_styles([
            {"selector": "table", "props": [
                ("border-collapse", "collapse"), ("width", "100%"),
                ("font-family", "Inter, monospace"), ("font-size", "12px"),
            ]},
            {"selector": "thead th", "props": [
                ("background-color", "#1E293B"), ("color", "#38BDF8"),
                ("font-weight", "600"), ("padding", "8px 14px"),
                ("text-align", "right"), ("border-bottom", "1px solid #334155"),
            ]},
            {"selector": "tbody td", "props": [
                ("background-color", "#0F172A"), ("color", "#CBD5E1"),
                ("padding", "6px 14px"), ("text-align", "right"),
                ("border-bottom", "1px solid #1E293B"),
            ]},
            {"selector": "tbody tr:nth-child(even) td", "props": [
                ("background-color", "#111827"),
            ]},
            {"selector": "tbody tr:hover td", "props": [
                ("background-color", "#1E3A5F"),
            ]},
        ])
        .format(precision=4)
    )
    display(HTML(header_html + styled.to_html()))

def _show_fig(fig, title: str) -> None:
    n = _next_figure()
    fig.update_layout(title_text=f"Figure {n}, {title}")
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

def insight(text: str) -> None:
    display(HTML(
        f"<div style='border-left:3px solid #38BDF8;padding:8px 16px;margin:4px 0 20px;"
        f"background:#0F172A;border-radius:0 8px 8px 0'>"
        f"<p style='margin:0;font-size:12px;color:#94A3B8;font-style:italic;"
        f"font-family:Inter,sans-serif'>💡 {text}</p></div>"
    ))

def kpi_cards(*cards: tuple) -> None:
    html_cards = []
    for card in cards:
        label = card[0]
        value = card[1]
        delta = (f"<p style='margin:4px 0 0;font-size:11px;color:#4ADE80;"
                 f"font-family:Inter,sans-serif'>{card[2]}</p>") if len(card) > 2 else ""
        html_cards.append(f"""
        <div style='background:{BG_CARD};border:1px solid #1E293B;border-radius:12px;
                    padding:20px 28px;min-width:160px;flex:1;text-align:center;
                    box-shadow:0 4px 24px rgba(0,0,0,.4)'>
            <p style='margin:0;font-size:10px;font-weight:600;letter-spacing:.1em;
                      text-transform:uppercase;color:#64748B;font-family:Inter,sans-serif'>{label}</p>
            <p style='margin:8px 0 0;font-size:30px;font-weight:700;
                      color:#F1F5F9;font-family:Inter,sans-serif'>{value}</p>
            {delta}
        </div>""")
    display(HTML(
        f"<div style='display:flex;gap:16px;flex-wrap:wrap;margin:16px 0'>"
        f"{''.join(html_cards)}</div>"
    ))




In [ ]:
# @title Load Data

def load_data(url: str = DATA_URL) -> pd.DataFrame:
    """Load the engine sensor dataset from a remote CSV."""
    return pd.read_csv(url)

df = load_data()
ProfileReport(df, title="Engine Sensor Report", explorative=True).to_file("engine_profile_report.html")

display_table(df.head(), "Raw Data Preview")
insight("First look at the data to check column names and types loaded correctly.")

kpi_cards(
    ("Rows",       f"{df.shape[0]:,}"),
    ("Columns",    df.shape[1]),
    ("Missing",    int(df.isnull().sum().sum())),
    ("Duplicates", int(df.duplicated().sum())),
)
insight("No missing values or duplicates, so the data goes straight into analysis without any cleaning.")




Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 6/6 [00:00<00:00, 12.20it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp
0,682,2.3917,4.6172,2.8490,76.2724,69.8846
1,605,5.4669,6.4244,5.7275,73.2227,74.9073
2,658,3.4342,3.6809,1.6787,88.0899,78.7048
3,749,2.0947,7.1209,1.6397,77.6616,82.3867
4,676,3.5382,5.9565,3.2253,75.2264,67.1532



##1.2. Exploratory Data Analysis (EDA)

Data inspection: .head(), .info(), .describe().

Check for missing or redundant records.

Visualization: Generate side-by-side Box Plots for all 6 engine features to visually confirm extreme operating conditions (Replicates Figure 1).

In [ ]:
# @title EDA: Dataset Summary

def summarize(df: pd.DataFrame) -> None:
    """Dataset info, descriptive stats, missing values, and central tendency."""
    info_df = pd.DataFrame({
        'Column':         df.columns,
        'Non-Null Count': df.notnull().sum().values,
        'Dtype':          df.dtypes.astype(str).values,
    })
    display_table(info_df, "Dataset Info")
    insight("All six columns are numeric (float64) with no nulls.")

    display_table(df.describe().T.round(4), "Descriptive Statistics")
    insight(
        "Looking at mean vs median helps spot skew. "
        "Engine rpm and Fuel pressure have higher means than medians, "
        "driven by occasional high-value spikes."
    )

    display_table(df.isnull().sum().rename("Missing").to_frame(), "Missing Values per Column")
    insight("All zeros, nothing to fill in.")

    central = pd.DataFrame({
        'Mean':   df[RAW_SENSORS].mean().round(4),
        'Median': df[RAW_SENSORS].median().round(4),
        'Skew':   df[RAW_SENSORS].skew().round(4),
        'Interpretation': [
            'Mean > Median — right-skewed; high-RPM bursts pull mean up',
            'Mean ≈ Median — near-symmetric; stable lubrication pressure',
            'Mean > Median — right-skewed; fuel demand spikes in acceleration',
            'Mean ≈ Median — stable coolant circuit pressure overall',
            'Mean ≈ Median — temperature distribution close to symmetric',
            'Mean ≈ Median — coolant temperature well-regulated',
        ]
    })
    display_table(central, "Mean vs Median, Central Tendency by Sensor")
    insight(
        "Engine rpm and Fuel pressure are the two right-skewed sensors. "
        "These are the sensors most likely to trigger IQR flags."
    )

summarize(df)




,Column,Non-Null Count,Dtype
0,Engine rpm,19535,int64
1,Lub oil pressure,19535,float64
2,Fuel pressure,19535,float64
3,Coolant pressure,19535,float64
4,lub oil temp,19535,float64
5,Coolant temp,19535,float64


,count,mean,std,min,25%,50%,75%,max
Engine rpm,19535.0000,791.2393,267.6112,61.0000,593.0000,746.0000,934.0000,2239.0000
Lub oil pressure,19535.0000,3.3038,1.0216,0.0034,2.5188,3.1620,4.0553,7.2656
Fuel pressure,19535.0000,6.6556,2.7610,0.0032,4.9169,6.2017,7.7450,21.1383
Coolant pressure,19535.0000,2.3354,1.0364,0.0025,1.6005,2.1669,2.8488,7.4785
lub oil temp,19535.0000,77.6434,3.1110,71.3220,75.7260,76.8174,78.0717,89.5808
Coolant temp,19535.0000,78.4274,6.2067,61.6733,73.8954,78.3467,82.9154,195.5279


,Missing
Engine rpm,0
Lub oil pressure,0
Fuel pressure,0
Coolant pressure,0
lub oil temp,0
Coolant temp,0


,Mean,Median,Skew,Interpretation
Engine rpm,791.2393,746.0000,0.9349,Mean > Median — right-skewed; high-RPM bursts pull mean up
Lub oil pressure,3.3038,3.1620,0.1958,Mean ≈ Median — near-symmetric; stable lubrication pressure
Fuel pressure,6.6556,6.2017,1.2164,Mean > Median — right-skewed; fuel demand spikes in acceleration
Coolant pressure,2.3354,2.1669,1.3094,Mean ≈ Median — stable coolant circuit pressure overall
lub oil temp,77.6434,76.8174,1.4964,Mean ≈ Median — temperature distribution close to symmetric
Coolant temp,78.4274,78.3467,0.4045,Mean ≈ Median — coolant temperature well-regulated


In [ ]:
# @title EDA: Histograms

def plot_histograms(df: pd.DataFrame, cols: list = RAW_SENSORS) -> None:
    """Histogram for each sensor in a 2-column grid."""
    n_cols = 2
    n_rows = -(-len(cols) // n_cols)
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=cols,
                        horizontal_spacing=0.1, vertical_spacing=0.12)
    for i, col in enumerate(cols):
        row = i // n_cols + 1
        col_pos = i % n_cols + 1
        fig.add_trace(
            go.Histogram(x=df[col], name=col, marker_color=C_NORMAL,
                         opacity=0.7, histnorm='probability density', showlegend=False),
            row=row, col=col_pos
        )
    fig.update_layout(**PLOTLY_LAYOUT, height=180 * n_rows, showlegend=False)
    _show_fig(fig, "Sensor Value Distributions")
    insight("Engine rpm and Fuel pressure show the most skew — they generate the most IQR flags.")

plot_histograms(df)




Modality & System States
- Unimodal: RPM, Fuel Pressure, Coolant Temp = steady setpoint.
- Bimodal: Lub Oil & Coolant Pressure = idle vs high load states.
- Skew/Low tails: pressure sensors show occasional off/noise states to watch.

In [ ]:
# @title  EDA: Box Plots

def plot_boxplots(df: pd.DataFrame) -> None:
    """Side-by-side box plots split by thermal and pressure groups."""
    thermal  = ['lub oil temp', 'Coolant temp']
    pressure = ['Lub oil pressure', 'Fuel pressure', 'Coolant pressure']
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Thermal Sensors (°C)", "Pressure Sensors (bar)"])
    for col in thermal:
        fig.add_trace(go.Box(x=df[col], name=col, marker_color=C_NORMAL,
                             line_color=C_NORMAL, fillcolor="rgba(56,189,248,0.12)",
                             boxmean=True), row=1, col=1)
    for col in pressure:
        fig.add_trace(go.Box(x=df[col], name=col, marker_color=C_ACCENT,
                             line_color=C_ACCENT, fillcolor="rgba(167,139,250,0.12)",
                             boxmean=True), row=1, col=2)
    fig.update_layout(**PLOTLY_LAYOUT, showlegend=False, height=420)
    _show_fig(fig, "Engine Sensor Distributions")
    insight("Pressure sensors have noticeably longer tails — they will contribute more IQR flags.")

plot_boxplots(df)

In [ ]:
# @title EDA: Correlation Heatmap

def plot_correlation(df: pd.DataFrame, cols: list = RAW_SENSORS) -> None:
    """Lower-triangle Pearson correlation heatmap."""
    corr        = df[cols].corr().round(2)
    mask        = np.triu(np.ones_like(corr, dtype=bool), k=1)
    corr_masked = corr.where(~mask)
    fig = go.Figure(go.Heatmap(
        z=corr_masked.values, x=corr.columns, y=corr.index,
        colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
        text=corr_masked.round(2).values,
        texttemplate="%{text}", textfont_size=10,
        hoverongaps=False,
        colorbar=dict(title="Pearson r", tickfont_color="#CBD5E1", title_font_color="#CBD5E1"),
    ))
    fig.update_layout(**PLOTLY_LAYOUT, height=480)
    _show_fig(fig, "Sensor Correlation Matrix")
    insight(
        "Some thermal channels are moderately correlated. "
        "Combining them into ratio features captures more signal than either alone."
    )

plot_correlation(df)






##1.3 Experimental Setup & Preprocessing

Define the target anomaly threshold (1% - 5%).

Standardize/Scale the data (specifically needed for the One-Class SVM).

In [ ]:
# @title Pre-processing: Scale Features

def scale(df: pd.DataFrame, cols: list) -> tuple:
    """Standardise selected columns to zero mean and unit variance."""
    scaler    = StandardScaler()
    out       = df.copy()
    out[cols] = scaler.fit_transform(df[cols])
    return out, scaler

df_raw            = df.copy()
df_scaled, scaler = scale(df, RAW_SENSORS)
features          = RAW_SENSORS

display_table(df_scaled[features].head(), "Scaled Features Preview")
insight(
    "After scaling, each sensor has mean 0 and std 1. "
    "Required for OCSVM and KMeans. "
    "Isolation Forest will use the original unscaled values."
)

,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp
0,-0.4082,-0.8928,-0.7383,0.4956,-0.4407,-1.3764
1,-0.6960,2.1173,-0.0838,3.2732,-1.4210,-0.5672
2,-0.4979,0.1277,-1.0774,-0.6336,3.3580,0.0447
3,-0.1578,-1.1835,0.1685,-0.6713,0.0059,0.6379
4,-0.4306,0.2295,-0.2532,0.8587,-0.7770,-1.8165




#2. Model 1: IQR-Based Detection (Baseline)

Calculate Q1, Q3, and IQR for each feature.

Define upper and lower bounds.

Create a loop to test anomaly counts based on the minimum number of simultaneously flagged features (1 through 5).

Output results into a DataFrame (Replicates Table 2).

In [ ]:
# @title Model 1.1 — IQR Functions
"""
IQR is a fast, interpretable, assumption-free baseline.
A row is flagged if >= 2 sensors simultaneously exceed Tukey fences.
"""

def calculate_outlier_flags(df_in, feature_list, iqr_multiplier=1.5):
    q1    = df_in[feature_list].quantile(0.25)
    q3    = df_in[feature_list].quantile(0.75)
    iqr   = q3 - q1
    lower = q1 - iqr_multiplier * iqr
    upper = q3 + iqr_multiplier * iqr
    display_table(pd.DataFrame(upper).T.round(4), "IQR Upper Bounds per Sensor")
    insight("Readings above these values trigger an outlier flag for that sensor.")
    display_table(pd.DataFrame(lower).T.round(4), "IQR Lower Bounds per Sensor")
    insight("Readings below these values trigger an outlier flag for that sensor.")
    flags         = ((df_in[feature_list] < lower) | (df_in[feature_list] > upper)).astype(int)
    flags.columns = [f"{c}_outlier" for c in feature_list]
    return flags

def aggregate_outlier_flags(flags, threshold=2):
    out              = flags.assign(Total_Flags=flags.sum(axis=1))
    out['iqr_anomaly'] = (out['Total_Flags'] >= threshold).astype(int)
    return out

def outlier_report(flags):
    counts = flags.drop(columns=['Total_Flags', 'iqr_anomaly'], errors='ignore').sum()
    pct    = (counts / len(flags)) * 100
    report = pd.DataFrame({'Count': counts, 'Percentage': pct})
    report.index = [i.replace('_outlier', '') for i in report.index]
    return report.round(2)

def threshold_sweep(flags, total):
    rows = []
    for t in range(1, 6):
        n = (flags['Total_Flags'] >= t).sum()
        rows.append({"Threshold": t, "Count": n,
                     "Percent": f"{(n / total) * 100:.2f}%",
                     "Assessment": "Optimal" if t == 2 else ""})
    return pd.DataFrame(rows)

def plot_flag_distribution(flags):
    counts         = flags['Total_Flags'].value_counts().sort_index().reset_index()
    counts.columns = ['Flags', 'Rows']
    fig = px.bar(counts, x='Flags', y='Rows',
                 labels={'Flags': 'Sensors Flagged Simultaneously', 'Rows': 'Number of Rows'},
                 color_discrete_sequence=[C_NORMAL])
    fig.add_vline(x=1.5, line_dash="dash", line_color=C_ANOMALY, line_width=2,
                  annotation_text="Anomaly cutoff (>= 2)", annotation_font_color=C_ANOMALY)
    fig.update_layout(**PLOTLY_LAYOUT, height=400)
    _show_fig(fig, "Distribution of Outlier Flags per Row")
    insight("Most rows sit at 0 or 1 flags — healthy operation. Bars to the right define the anomaly set.")




In [ ]:
# @title  Model 1 — IQR: Flags & Threshold Sweep

flags_df   = calculate_outlier_flags(df_scaled, features)
outlier_df = aggregate_outlier_flags(flags_df, threshold=2)
df_scaled['iqr_anomaly'] = outlier_df['iqr_anomaly']

kpi_cards(
    ("Total Rows",    f"{len(df_scaled):,}"),
    ("IQR Anomalies", int(outlier_df['iqr_anomaly'].sum()),
     f"{outlier_df['iqr_anomaly'].mean()*100:.2f}%"),
)
display_table(outlier_report(outlier_df), "Feature Outlier Report")
insight("Engine rpm and Fuel pressure have the highest individual outlier rates.")

display_table(threshold_sweep(outlier_df, len(df_scaled)), "Threshold Sweep (1-5 Sensors)")
insight(
    "Threshold 2 gives ~1.6% anomaly rate, within the expected 1-5% range. "
    "Requiring two sensors to flag the same row cuts down on single-sensor noise."
)




,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp
0,2.4449,2.9915,1.9310,2.3023,1.2687,2.9030


,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp
0,-2.6522,-3.0243,-2.1662,-2.5160,-1.7474,-2.9101


,Count,Percentage
Engine rpm,464,2.3800
Lub oil pressure,66,0.3400
Fuel pressure,1135,5.8100
Coolant pressure,785,4.0200
lub oil temp,2617,13.4000
Coolant temp,2,0.0100


,Threshold,Count,Percent,Assessment
0,1,4636,23.73%,
1,2,422,2.16%,Optimal
2,3,11,0.06%,
3,4,0,0.00%,
4,5,0,0.00%,


In [ ]:
# @title Model 1.2 — IQR: Review Flagged Rows

display_table(
    df_scaled[RAW_SENSORS].head().join(outlier_df[['Total_Flags', 'iqr_anomaly']].head()),
    "Sensor Values with Outlier Flags"
)
insight("This shows how the per-sensor binary flags add up to the final 0/1 anomaly label.")

binary_display = flags_df.head(10).copy()
binary_display.columns = [c.replace('_outlier', '') for c in binary_display.columns]
display_table(binary_display, "Per-Feature Binary Outlier Columns (first 10 rows)")
insight("Each column is 1 if that sensor reading was outside the IQR fence, 0 if not.")

display_table(
    df_raw.loc[outlier_df['iqr_anomaly'] == 1, features].head(),
    "Top 5 IQR Anomalies, Original Sensor Values"
)
insight("Actual unscaled readings for the first five flagged rows.")



,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp,Total_Flags,iqr_anomaly
0,-0.4082,-0.8928,-0.7383,0.4956,-0.4407,-1.3764,0,0
1,-0.6960,2.1173,-0.0838,3.2732,-1.4210,-0.5672,1,0
2,-0.4979,0.1277,-1.0774,-0.6336,3.3580,0.0447,1,0
3,-0.1578,-1.1835,0.1685,-0.6713,0.0059,0.6379,0,0
4,-0.4306,0.2295,-0.2532,0.8587,-0.7770,-1.8165,0,0


,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp
0,0,0,0,0,0,0
1,0,0,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0
5,0,0,0,0,0,0
6,0,0,1,0,0,0
7,0,0,1,0,0,0
8,0,0,0,0,1,0
9,0,0,1,0,0,0


,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp
113,1495,3.2762,3.7144,2.4953,85.0532,75.7430
122,1454,2.0634,6.1683,1.2601,83.3723,82.7182
131,830,3.2319,13.4792,2.5681,87.4804,73.4282
144,1762,1.6975,4.3278,2.5234,86.7503,82.3932
148,1494,2.4432,3.5481,1.9534,82.2345,83.9539


In [ ]:
# @title  Model 1.3 — IQR: Chart & Observations

plot_flag_distribution(outlier_df)

display(HTML("""
<div style='background:#0F172A;border:1px solid #334155;border-radius:10px;
            padding:20px 24px;margin:16px 0'>
  <p style='margin:0 0 10px;font-size:12px;font-weight:700;letter-spacing:.06em;
            text-transform:uppercase;color:#38BDF8;font-family:Inter,sans-serif'>
    IQR Method — Observations
  </p>
  <ul style='margin:0;padding-left:18px;color:#CBD5E1;font-family:Inter,sans-serif;
             font-size:12px;line-height:1.9'>
    <li><b>What works well:</b> Fast, easy to explain, needs no labelled data.
        You can tell a maintenance engineer exactly which sensor crossed which threshold.</li>
    <li><b>Threshold choice:</b> Threshold=2 gave ~1.6% anomaly rate, within the 1-5% band.</li>
    <li><b>Main limitation:</b> IQR looks at each sensor on its own. RPM slightly elevated
        AND oil pressure slightly low at the same time might not trigger any flags individually,
        even though the combination is concerning. That is why we need the ML models.</li>
    <li><b>Overall:</b> Good for quick alerts on obvious single-sensor extremes,
        but not sufficient on its own.</li>
  </ul>
</div>
"""))




#3. Model 2: One-Class SVM

Initialize OneClassSVM with RBF kernel on scaled data.

Perform grid search/parameter tuning for combinations of nu (0.01, 0.03, 0.05) and gamma (0.1, 0.5, 0.7).

Output tuning results into a DataFrame showing anomaly counts and percentages (Replicates Table 3).

In [ ]:
# @title  Model 2.1 — OCSVM Functions
"""
OCSVM learns a non-linear boundary in kernel space enclosing the normal region.
Captures multivariate structure — a reading can be normal on each sensor
individually but anomalous in combination.
"""

def run_ocsvm(df, cols, nu=0.03, gamma='scale'):
    model = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
    return model.fit_predict(df[cols]), model

def ocsvm_grid_search(X, nu_values, gamma_values, feature_names):
    rows = []
    for nu in nu_values:
        for gamma in gamma_values:
            preds, _ = run_ocsvm(pd.DataFrame(X, columns=feature_names), feature_names, nu, gamma)
            n = (preds == -1).sum()
            rows.append({"Nu": nu, "Gamma": gamma, "Anomaly Count": n,
                         "Percentage (%)": f"{n / len(X) * 100:.2f}%"})
    return pd.DataFrame(rows)

def plot_ocsvm_grid(X_pca, nu_values, gamma_values):
    titles = [f"nu={nu} | g={g}" for nu in nu_values for g in gamma_values]
    fig    = make_subplots(rows=len(nu_values), cols=len(gamma_values),
                           subplot_titles=titles,
                           horizontal_spacing=0.05, vertical_spacing=0.08)
    for i, nu in enumerate(nu_values):
        for j, gamma in enumerate(gamma_values):
            preds, _ = run_ocsvm(pd.DataFrame(X_pca, columns=['pc1', 'pc2']),
                                 ['pc1', 'pc2'], nu, gamma)
            for label, color, name in [(1, C_NORMAL, "Normal"), (-1, C_ANOMALY, "Anomaly")]:
                mask = preds == label
                fig.add_trace(go.Scatter(
                    x=X_pca[mask, 0], y=X_pca[mask, 1], mode='markers', name=name,
                    marker=dict(color=color, size=3 if label == 1 else 6,
                                opacity=0.3 if label == 1 else 0.7),
                    showlegend=(i == 0 and j == 0),
                ), row=i + 1, col=j + 1)
    fig.update_layout(**PLOTLY_LAYOUT, height=900)
    _show_fig(fig, "One-Class SVM, Boundary Sensitivity (nu x gamma)")
    insight("Higher nu expands the anomaly region. nu=0.03, gamma=0.5 gives the best-balanced boundary.")




In [ ]:
# @title  Model 2.2 — OCSVM: Grid Search

X_scaled_arr = df_scaled[features].values
X_pca_tune   = PCA(n_components=2).fit_transform(X_scaled_arr)

display_table(
    ocsvm_grid_search(X_scaled_arr, NU_VALUES, GAMMA_VALUES, features),
    "OCSVM Parameter Tuning Grid (nu x gamma)"
)
insight("nu=0.03 gives roughly 3% anomalies. Gamma affects boundary shape rather than total count.")
plot_ocsvm_grid(X_pca_tune, NU_VALUES, GAMMA_VALUES)




,Nu,Gamma,Anomaly Count,Percentage (%)
0,0.0100,0.1000,189,0.97%
1,0.0100,0.5000,611,3.13%
2,0.0100,0.7000,1067,5.46%
3,0.0300,0.1000,586,3.00%
4,0.0300,0.5000,737,3.77%
5,0.0300,0.7000,1075,5.50%
6,0.0500,0.1000,978,5.01%
7,0.0500,0.5000,1043,5.34%
8,0.0500,0.7000,1234,6.32%


KeyboardInterrupt: 

In [ ]:
# @title  Model 2.3 — OCSVM: Final Model & Observations

ocsvm_preds, ocsvm_model = run_ocsvm(
    pd.DataFrame(X_scaled_arr, columns=features), features, nu=TARGET_NU, gamma=0.5
)
kpi_cards(("OCSVM Anomalies", int((ocsvm_preds == -1).sum()),
           f"{(ocsvm_preds == -1).mean()*100:.2f}%"))

display(HTML("""
<div style='background:#0F172A;border:1px solid #334155;border-radius:10px;
            padding:20px 24px;margin:16px 0'>
  <p style='margin:0 0 10px;font-size:12px;font-weight:700;letter-spacing:.06em;
            text-transform:uppercase;color:#38BDF8;font-family:Inter,sans-serif'>
    One-Class SVM — Chosen Configuration & Observations
  </p>
  <ul style='margin:0;padding-left:18px;color:#CBD5E1;font-family:Inter,sans-serif;
             font-size:12px;line-height:1.9'>
    <li><b>Selected:</b> nu=0.03, gamma=0.5, kernel=RBF. Anomaly rate ~3%, within 1-5%.</li>
    <li><b>nu:</b> Upper bound on the outlier fraction. 0.03 maps directly to the 3% assumption.</li>
    <li><b>gamma:</b> 0.5 follows the normal cluster closely without fragmenting at extremes.</li>
    <li><b>Effectiveness:</b> Catches multivariate anomalies invisible to IQR.</li>
    <li><b>Limitation:</b> Computationally expensive and requires feature scaling.</li>
  </ul>
</div>
"""))






#4. Model 3: Isolation Forest

Initialize IsolationForest on unscaled data.

Test different contamination values (0.01, 0.03, 0.05).

Output tuning results into a DataFrame (Replicates Table 4).

In [ ]:
# @title Model 3.1 — IF Functions
"""
Isolation Forest detects anomalies by random partitioning. Points requiring fewer
splits to isolate score as anomalies. Robust to non-Gaussian distributions.
Does not require feature scaling.
"""

def run_isolation_forest(X, contamination=0.03, random_state=42):
    model = IsolationForest(contamination=contamination, random_state=random_state)
    return model.fit_predict(X), model

def iso_sensitivity(X, contam_levels):
    rows = []
    for c in contam_levels:
        preds, _ = run_isolation_forest(X, contamination=c)
        n = (preds == -1).sum()
        rows.append({"Contamination": c, "Anomaly Count": n,
                     "Percentage (%)": f"{n / len(X) * 100:.2f}%",
                     "Assessment": "Optimal" if c == 0.03 else "Trial"})
    return pd.DataFrame(rows)

def plot_iso_sensitivity(X_pca, X_raw, contam_levels):
    fig = make_subplots(rows=1, cols=len(contam_levels),
                        subplot_titles=[f"Contamination: {c}" for c in contam_levels],
                        horizontal_spacing=0.06)
    for j, c in enumerate(contam_levels):
        preds, _ = run_isolation_forest(X_raw, contamination=c)
        for label, color, name in [(1, C_NORMAL, "Normal"), (-1, C_ANOMALY, "Anomaly")]:
            mask = preds == label
            fig.add_trace(go.Scatter(
                x=X_pca[mask, 0], y=X_pca[mask, 1], mode='markers', name=name,
                marker=dict(color=color, size=4 if label == 1 else 7,
                            opacity=0.35 if label == 1 else 0.8),
                showlegend=(j == 0),
            ), row=1, col=j + 1)
    fig.update_layout(**PLOTLY_LAYOUT, height=460)
    _show_fig(fig, "Isolation Forest, Contamination Sensitivity")
    insight("A stable red core across all three panels = high-confidence anomalies.")




In [ ]:
# @title  Model 3.2 — IF: Sensitivity Analysis

X_raw_arr = df_raw[features].values
X_pca_raw = PCA(n_components=2).fit_transform(X_raw_arr)

display_table(iso_sensitivity(X_raw_arr, CONTAM_LEVELS), "Isolation Forest Sensitivity Analysis")
insight("0.03 aligns with the ~3% fault rate assumption and matches OCSVM for fair comparison.")
plot_iso_sensitivity(X_pca_raw, X_raw_arr, CONTAM_LEVELS)




In [ ]:
# @title  Model 3.3 — IF: Final Model & Observations

if_preds, if_model = run_isolation_forest(X_raw_arr, contamination=TARGET_NU)
kpi_cards(("IF Anomalies", int((if_preds == -1).sum()),
           f"{(if_preds == -1).mean()*100:.2f}%"))

display(HTML("""
<div style='background:#0F172A;border:1px solid #334155;border-radius:10px;
            padding:20px 24px;margin:16px 0'>
  <p style='margin:0 0 10px;font-size:12px;font-weight:700;letter-spacing:.06em;
            text-transform:uppercase;color:#38BDF8;font-family:Inter,sans-serif'>
    Isolation Forest — Chosen Configuration & Observations
  </p>
  <ul style='margin:0;padding-left:18px;color:#CBD5E1;font-family:Inter,sans-serif;
             font-size:12px;line-height:1.9'>
    <li><b>Selected:</b> contamination=0.03, n_estimators=100, random_state=42.</li>
    <li><b>contamination:</b> Mirrors nu in OCSVM — enables a direct comparison.</li>
    <li><b>No scaling required:</b> Partitions on random split points, not distances.</li>
    <li><b>Effectiveness:</b> A stable anomaly core at 0.01 persists at 0.03 and 0.05
        — these are the highest-confidence anomalies.</li>
    <li><b>Advantage over IQR:</b> Detects cross-sensor interactions without manual thresholds.</li>
  </ul>
</div>
"""))






#5. PCA Visualization & Model Comparison

Apply PCA(n_components=2) to project the high-dimensional feature space onto 2D.

Plot side-by-side scatter plots for One-Class SVM and Isolation Forest anomalies in the PCA space (Replicates Figure 2).


In [ ]:
# @title  5.1 — PCA Functions

def plot_pca(X, labels, title="PCA Anomaly Map"):
    """Project data to 2D via PCA and colour points by anomaly label."""
    coords  = PCA(n_components=2).fit_transform(X)
    labels  = np.asarray(labels)
    df_plot = pd.DataFrame({
        "PC1":  coords[:, 0], "PC2": coords[:, 1],
        "Type": np.where(labels == 1, "Normal", "Anomaly"),
    })
    fig = px.scatter(df_plot, x="PC1", y="PC2", color="Type",
                     color_discrete_map={"Normal": C_NORMAL, "Anomaly": C_ANOMALY},
                     opacity=0.6,
                     labels={"PC1": "PC1 — Thermal Load",
                             "PC2": "PC2 — Engine Speed / Oil Pressure"})
    fig.update_traces(marker_size=5)
    fig.update_traces(marker_size=9, selector=dict(name="Anomaly"))
    fig.update_layout(**PLOTLY_LAYOUT, height=480)
    _show_fig(fig, title)
    insight("Well-separated red clusters confirm anomalies occupy a distinct region.")

def plot_model_comparison(X_pca, preds_ocsvm, preds_iso):
    """Side-by-side PCA scatter comparing OCSVM and IF."""
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[f"One-Class SVM (nu={TARGET_NU})",
                                        f"Isolation Forest (contam={TARGET_NU})"],
                        horizontal_spacing=0.08)
    for j, preds in enumerate([preds_ocsvm, preds_iso]):
        for label, color, name in [(1, C_NORMAL, "Normal"), (-1, C_ANOMALY, "Anomaly")]:
            mask = np.asarray(preds) == label
            fig.add_trace(go.Scatter(
                x=X_pca[mask, 0], y=X_pca[mask, 1], mode='markers', name=name,
                marker=dict(color=color, size=4 if label == 1 else 8,
                            opacity=0.35 if label == 1 else 0.85),
                showlegend=(j == 0),
            ), row=1, col=j + 1)
    fig.update_layout(**PLOTLY_LAYOUT, height=500)
    _show_fig(fig, "Model Comparison, OCSVM vs Isolation Forest (PCA Projection)")
    insight("Anomalies in both panels = high-confidence faults.")




In [ ]:
# @title 5.2 — PCA: Individual Model Maps

plot_pca(X_scaled_arr, ocsvm_preds, "OCSVM, Anomaly Map")
plot_pca(X_raw_arr,    if_preds,    "Isolation Forest, Anomaly Map")




In [ ]:
# @title 5.3 — PCA: Loadings (what do the axes mean?)

pca_shared   = PCA(n_components=2)
X_pca_shared = pca_shared.fit_transform(X_scaled_arr)

loadings_raw = pd.DataFrame(
    pca_shared.components_.T, index=RAW_SENSORS, columns=['PC1', 'PC2']
).round(3)

display_table(
    loadings_raw.assign(
        PC1_abs=loadings_raw['PC1'].abs(),
        PC2_abs=loadings_raw['PC2'].abs()
    ).sort_values('PC1_abs', ascending=False).drop(columns=['PC1_abs', 'PC2_abs']),
    "PCA Loadings — 6 Raw Features"
)
insight(
    "The features with the highest absolute loading on PC1 define what the horizontal axis "
    "actually represents. Use these to interpret where anomalies cluster."
)




In [ ]:
# @title 5.4 — PCA: Side-by-Side Model Comparison

plot_model_comparison(X_pca_shared, ocsvm_preds, if_preds)



In [ ]:
# @title  5.5 — PCA: Effectiveness & Model Agreement

display(HTML("""
<div style='background:#0F172A;border:1px solid #334155;border-radius:10px;
            padding:20px 24px;margin:16px 0'>
  <p style='margin:0 0 10px;font-size:12px;font-weight:700;letter-spacing:.06em;
            text-transform:uppercase;color:#38BDF8;font-family:Inter,sans-serif'>
    PCA Visualisation — Effectiveness in Highlighting Outliers
  </p>
  <ul style='margin:0;padding-left:18px;color:#CBD5E1;font-family:Inter,sans-serif;
             font-size:12px;line-height:1.9'>
    <li><b>OCSVM:</b> Anomalies cluster tightly at the periphery — physically meaningful extremes, not noise.</li>
    <li><b>IF:</b> Anomalies more broadly distributed, reflecting sensitivity to low-density micro-regions.</li>
    <li><b>Model agreement:</b> Red in both panels = high-confidence multi-method anomaly.</li>
    <li><b>Limitation:</b> 2D projection loses some variance. For interpretation only.</li>
  </ul>
</div>
"""))

agreement  = int(((ocsvm_preds == -1) & (if_preds == -1)).sum())
ocsvm_only = int(((ocsvm_preds == -1) & (if_preds == 1)).sum())
if_only    = int(((ocsvm_preds == 1)  & (if_preds == -1)).sum())

kpi_cards(
    ("Both Agree (Anomaly)", agreement,   f"{agreement/len(ocsvm_preds)*100:.2f}%"),
    ("OCSVM Only",           ocsvm_only,  f"{ocsvm_only/len(ocsvm_preds)*100:.2f}%"),
    ("IF Only",              if_only,     f"{if_only/len(if_preds)*100:.2f}%"),
)
insight(
    "Readings flagged by both models are the highest-confidence anomalies. "
    "OCSVM-only flags tend to be boundary cases; IF-only flags are isolated low-density points."
)





#6. Feature Engineering for Improved Diagnostic Accuracy

1. Problem Statement
   * Current: time-based maintenance misses real stress → downtime + failures.
   * Goal: flag anomalous sensor patterns early to avoid costly at-sea failures.

   Business Impact
   * Financial: 20k–50k/day downtime; penalties for late delivery.
   * Safety: loss of propulsion → fire/toxic risk.
   * Operational: emergency towing/unscheduled port ≈ 10x planned intervention.

2. Hypothesis
   * H0: Unsupervised anomaly detection does not reduce downtime/costs.
   * H1: Multi-sensor anomaly detection cuts downtime/emergency repair cost.



3. Key Performance Indicators (KPIs)
   * MTTD: detect earlier than manual checks; keep alerts 1–5% (contamination).
   * False Discovery Rate: minimize noise; Cost Variance: predicted vs emergency repair delta.

4. Proposed Approach
   * Features: ratios/power terms to capture sensor interactions.
   * Baseline: IQR needs 2+ sensors out simultaneously.
   * ML: Isolation Forest + One-Class SVM at ~3% contamination; visualised with PCA.
   * Feature importance analysis (absolute mean difference between healthy and critical
     states) identified lub_efficiency and power_intensity as the strongest
     discriminators between normal and anomalous engine operation.

   Why 12 features help
   * The 6 raw sensors catch limit breaks (hard excursions where a single sensor
     breaches its normal range).
   * The 6 engineered ratios catch relational imbalances between load, pressure, and
     temperature — degradation patterns that develop before any single sensor
     breaches its individual limit.
   * This combined 12-feature space improves separability and reduces false alarms
     by normalising co-movements and exposing early-stage degradation.

   Data Scaling Strategy: Z-Score Standardisation
   * Standardise to keep outlier magnitude and equalise sensor scales for SVM/PCA/KMeans.
   * Prevents min-max squashing; preserves distance-based sensitivity for anomalies.


In [ ]:
# @title 6.1 — Rationale & Sensor Themes

themes = pd.DataFrame([
    ("Combustion Efficiency", "RPM vs. Fuel Pressure",
     "High RPM without proportional fuel increase suggests internal drag or resistance."),
    ("Friction Management",   "Oil Pressure / Temp Ratio",
     "A drop in this ratio signals thinning lubricant and potential seizure."),
    ("Thermal Regulation",    "Coolant Pressure vs. Temp",
     "High Temp + Low Pressure indicates a leak or pump failure."),
    ("Injector Health",       "Fuel Pressure / RPM",
     "Disproportionately high fuel demand at low RPM signals injector blockage."),
], columns=["Theme", "Sensor Interaction", "Business Logic"])

display_table(themes, "Sensor Themes & Engineering Rationale")




In [ ]:
# @title  6.2 — Build Engineered Features

class FeatureAdder(BaseEstimator, TransformerMixin):
    """Sklearn-compatible transformer that appends six engineered features."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = pd.DataFrame(X, columns=RAW_SENSORS)
        df['lub_efficiency']    = df['Lub oil pressure'] / (df['lub oil temp'] + 1e-6)
        df['cooling_delta']     = df['Coolant pressure'] - df['Coolant temp']
        df['power_intensity']   = df['Fuel pressure'] / (df['Engine rpm'] + 1e-6)
        df['rpm_squared']       = df['Engine rpm'] ** 2
        df['log_fuel_pressure'] = np.log1p(df['Fuel pressure'])
        df['sqrt_rpm']          = np.sqrt(df['Engine rpm'])
        return df[RAW_SENSORS + ENGINEERED_FEATURES].values

engine_df = df_raw.copy()
for col, expr in [
    ('lub_efficiency',    lambda d: d['Lub oil pressure'] / (d['lub oil temp'] + 1e-6)),
    ('cooling_delta',     lambda d: d['Coolant pressure'] - d['Coolant temp']),
    ('power_intensity',   lambda d: d['Fuel pressure'] / (d['Engine rpm'] + 1e-6)),
    ('rpm_squared',       lambda d: d['Engine rpm'] ** 2),
    ('log_fuel_pressure', lambda d: np.log1p(d['Fuel pressure'])),
    ('sqrt_rpm',          lambda d: np.sqrt(d['Engine rpm'])),
]:
    engine_df[col] = expr(engine_df)

all_features = RAW_SENSORS + ENGINEERED_FEATURES
X_eng        = StandardScaler().fit_transform(engine_df[all_features])
X_eng_raw    = engine_df[all_features].values
pca_eng      = PCA(n_components=2)
X_eng_pca    = pca_eng.fit_transform(X_eng)

display_table(engine_df[all_features].head(), "Engineered Feature Set Preview")
insight(
    "The six new columns encode physical relationships between sensors. "
    "lub_efficiency captures lubrication health; power_intensity captures combustion load."
)




In [ ]:
# @title  6.3 — PCA Loadings for 12 Engineered Features

loadings_eng = pd.DataFrame(
    pca_eng.components_.T, index=all_features, columns=['PC1', 'PC2']
).round(3)

display_table(
    loadings_eng.assign(
        PC1_abs=loadings_eng['PC1'].abs(),
        PC2_abs=loadings_eng['PC2'].abs()
    ).sort_values('PC1_abs', ascending=False).drop(columns=['PC1_abs', 'PC2_abs']),
    "PCA Feature Loadings — 12 Engineered Features"
)
insight("Use these loadings to label the axes in the engineered-feature PCA plots accurately.")




In [ ]:
# @title  6.4 — Run All 4 Models on 12 Features

# IQR
eng_iqr_flags = pd.DataFrame(index=engine_df.index)
for col in all_features:
    q1, q3 = engine_df[col].quantile(0.25), engine_df[col].quantile(0.75)
    iqr    = q3 - q1
    eng_iqr_flags[f'{col}_out'] = (
        (engine_df[col] < q1 - 1.5 * iqr) | (engine_df[col] > q3 + 1.5 * iqr)
    ).astype(int)
engine_df['final_iqr_anomaly'] = (eng_iqr_flags.sum(axis=1) >= 2).astype(int)

# KMeans
kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
engine_df['cluster']       = kmeans.fit_predict(X_eng)
centroids                  = kmeans.cluster_centers_
engine_df['centroid_dist'] = [
    np.linalg.norm(X_eng[i] - centroids[engine_df['cluster'].iloc[i]])
    for i in range(len(engine_df))
]
c_stats = engine_df.groupby('cluster')['centroid_dist'].agg(['mean', 'std'])
engine_df['final_kmeans_anomaly'] = engine_df.apply(
    lambda r: int(abs(r['centroid_dist'] - c_stats.loc[r['cluster'], 'mean'])
                  > 2 * c_stats.loc[r['cluster'], 'std']), axis=1
)

# IF on unscaled
engine_df['final_iso_anomaly'] = np.where(
    IsolationForest(contamination=TARGET_NU, random_state=RANDOM_STATE)
    .fit_predict(X_eng_raw) == -1, 1, 0)

# OCSVM with tuned gamma=0.5
engine_df['final_ocsvm_anomaly'] = np.where(
    OneClassSVM(kernel='rbf', nu=TARGET_NU, gamma=0.5).fit_predict(X_eng) == -1, 1, 0)

# Consensus
triple = (
    engine_df[['final_iqr_anomaly', 'final_iso_anomaly', 'final_ocsvm_anomaly']] == 1
).all(axis=1)

engine_df['critical_anomaly'] = (
    (engine_df['final_iso_anomaly'] == 1) & (engine_df['final_ocsvm_anomaly'] == 1)
).astype(int)

kpi_cards(
    ("IQR Anomalies",    int(engine_df['final_iqr_anomaly'].sum()),
     f"{engine_df['final_iqr_anomaly'].mean()*100:.2f}%"),
    ("KMeans Anomalies", int(engine_df['final_kmeans_anomaly'].sum()),
     f"{engine_df['final_kmeans_anomaly'].mean()*100:.2f}%"),
    ("OCSVM Anomalies",  int(engine_df['final_ocsvm_anomaly'].sum()),
     f"{engine_df['final_ocsvm_anomaly'].mean()*100:.2f}%"),
    ("IF Anomalies",     int(engine_df['final_iso_anomaly'].sum()),
     f"{engine_df['final_iso_anomaly'].mean()*100:.2f}%"),
)
kpi_cards(
    ("Triple Consensus",     int(triple.sum()),     f"{triple.mean()*100:.2f}%"),
    ("IF + OCSVM Consensus", int(engine_df['critical_anomaly'].sum()),
     f"{engine_df['critical_anomaly'].mean()*100:.2f}%"),
)
insight(
    "Triple consensus (IQR + IF + OCSVM) gives the highest-confidence fault set. "
    "IF + OCSVM is used for impact analysis — less sensitive to the IQR threshold choice."
)




In [ ]:
# @title  6.5 — 4-Model Visual Comparison

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.patch.set_facecolor('#0F172A')

methods = [
    ('K-Means (Centroid Distance)', 'final_kmeans_anomaly'),
    ('IQR (12 features)',           'final_iqr_anomaly'),
    ('One-Class SVM (Kernel)',       'final_ocsvm_anomaly'),
    ('Isolation Forest (Tree)',      'final_iso_anomaly'),
]

for ax, (name, col) in zip(axes, methods):
    ax.set_facecolor('#1E293B')
    normal  = engine_df[col] == 0
    anomaly = engine_df[col] == 1
    ax.scatter(X_eng_pca[normal,  0], X_eng_pca[normal,  1], c='#38BDF8', alpha=0.25, s=8,  label='Normal')
    ax.scatter(X_eng_pca[anomaly, 0], X_eng_pca[anomaly, 1], c='#F87171', alpha=0.80, s=16, label='Anomaly')
    ax.set_title(name, color='#F1F5F9', fontsize=10, pad=8)
    ax.tick_params(colors='#64748B', labelsize=7)
    for spine in ax.spines.values():
        spine.set_color('#334155')
    n_anom = int(anomaly.sum())
    ax.set_xlabel(f'{n_anom} anomalies ({n_anom/len(engine_df)*100:.1f}%)', color='#64748B', fontsize=8)
    ax.legend(fontsize=7, facecolor='#0F172A', labelcolor='#CBD5E1', edgecolor='#334155', markerscale=1.5)

fig.suptitle("4-Model Comparison on Engineered Features (PCA Projection)", color='#F1F5F9', fontsize=13, y=1.02)
plt.tight_layout()
display(fig)
plt.close()
insight("Readings red in all four panels are the most critical anomalies.")




In [ ]:
# @title  6.6 — Anomaly Density Heatmap

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0F172A')

ax = axes[0]
ax.set_facecolor('#0F172A')
h = ax.hist2d(X_eng_pca[:, 0], X_eng_pca[:, 1], bins=80, cmap='Blues',
              norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(h[3], ax=ax, label='Log count (normal density)')
ax.set_title('Normal Operating Region\n(all 19,535 readings)', color='#F1F5F9', fontsize=11)
ax.set_xlabel('PC1 — Engine Speed (RPM)', color='#64748B', fontsize=9)
ax.set_ylabel('PC2 — Fuel Demand', color='#64748B', fontsize=9)
ax.tick_params(colors='#64748B', labelsize=7)
for spine in ax.spines.values():
    spine.set_color('#334155')

ax2 = axes[1]
ax2.set_facecolor('#0F172A')
consensus_mask = engine_df['critical_anomaly'] == 1
X_anom = X_eng_pca[consensus_mask]
ax2.hist2d(X_eng_pca[:, 0], X_eng_pca[:, 1], bins=80, cmap='Blues', alpha=0.25,
           norm=plt.matplotlib.colors.LogNorm())
h2 = ax2.hist2d(X_anom[:, 0], X_anom[:, 1], bins=40, cmap='Reds', alpha=0.85,
                norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(h2[3], ax=ax2, label='Log count (anomaly density)')
n_anom = int(consensus_mask.sum())
ax2.set_title(f'Consensus Anomaly Cluster\n({n_anom} readings, {n_anom/len(engine_df)*100:.1f}%)',
              color='#F1F5F9', fontsize=11)
ax2.set_xlabel('PC1 — Engine Speed (RPM)', color='#64748B', fontsize=9)
ax2.set_ylabel('PC2 — Fuel Demand', color='#64748B', fontsize=9)
ax2.tick_params(colors='#64748B', labelsize=7)
for spine in ax2.spines.values():
    spine.set_color('#334155')

fig.suptitle('Where Do Anomalies Cluster? Normal Density vs Consensus Fault Density',
             color='#F1F5F9', fontsize=13, y=1.02)
plt.tight_layout()
display(fig)
plt.close()
insight(
    "Anomalies cluster in one corner rather than scattering randomly. "
    "This confirms the models are detecting a real operating pattern, not noise."
)




In [ ]:
# @title  6.7 — Safety Zone (OCSVM Decision Boundary)

def plot_safety_zone(X_pca, anomaly_labels, nu=TARGET_NU):
    """OCSVM decision boundary contour overlaid on PCA scatter."""
    viz_svm = OneClassSVM(kernel='rbf', gamma=0.1, nu=nu).fit(X_pca)
    x_ax    = np.linspace(X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1, CONTOUR_RES)
    y_ax    = np.linspace(X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1, CONTOUR_RES)
    xx, yy  = np.meshgrid(x_ax, y_ax)
    Z       = viz_svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    labels  = np.asarray(anomaly_labels)
    fig     = go.Figure()
    fig.add_trace(go.Contour(
        x=x_ax, y=y_ax, z=Z,
        colorscale=[[0, "#0F172A"], [0.5, "#1E3A5F"], [1, "#38BDF8"]],
        contours=dict(start=Z.min(), end=0, size=(0 - Z.min()) / 6),
        showscale=False, opacity=0.5,
    ))
    fig.add_trace(go.Contour(
        x=x_ax, y=y_ax, z=Z,
        contours=dict(type="constraint", operation="=", value=0),
        line=dict(color=C_ANOMALY, width=2, dash="dot"),
        showscale=False, name="Decision boundary",
    ))
    for lbl, color, name, sym in [(0, C_NORMAL, "Normal", "circle"),
                                   (1, C_ANOMALY, "Anomaly", "x")]:
        mask = labels == lbl
        fig.add_trace(go.Scatter(
            x=X_pca[mask, 0], y=X_pca[mask, 1], mode='markers', name=name,
            marker=dict(color=color, size=5 if lbl == 0 else 9,
                        opacity=0.5 if lbl == 0 else 0.9, symbol=sym),
        ))
    fig.update_layout(**PLOTLY_LAYOUT, height=560)
    _show_fig(fig, "Engine Operational Safety Zone (Engineered Features)")
    insight("Points inside the blue zone are normal. Anomalies outside the dotted line = highest priority.")

plot_safety_zone(X_eng_pca, engine_df['final_ocsvm_anomaly'].values)




In [ ]:
# @title  6.8 — Impact Analysis: Healthy vs Critical

def plot_feature_importance(impact_df):
    """Horizontal bar chart of top 12 features by |Critical - Healthy|."""
    diffs = (impact_df.loc['Critical avg'] - impact_df.loc['Healthy avg']).abs()
    top   = diffs.sort_values(ascending=True).tail(12)
    fig = px.bar(
        x=top.values, y=top.index, orientation='h',
        labels={"x": "Absolute Mean Difference", "y": ""},
        color=top.values,
        color_continuous_scale=[[0, C_NORMAL], [0.5, C_ACCENT], [1, C_ANOMALY]],
    )
    fig.update_layout(**PLOTLY_LAYOUT, coloraxis_showscale=False, height=460)
    _show_fig(fig, "Feature Importance, |Critical - Healthy|")
    insight("Engineered features like lub_efficiency and power_intensity outrank raw sensors.")

impact_table = (
    engine_df.groupby('critical_anomaly')[all_features]
    .mean()
    .rename(index={0: 'Healthy avg', 1: 'Critical avg'})
)
display_table(impact_table.T.round(4), "Healthy vs Critical, Mean Feature Values")
insight("Features with the biggest gap between healthy and critical averages are the most diagnostic.")
plot_feature_importance(impact_table)




In [ ]:
# @title  6.9 — Maintenance Priority Table

priority_table = (
    engine_df[engine_df['critical_anomaly'] == 1]
    .assign(Severity=lambda x: x['centroid_dist'])
    .sort_values('Severity', ascending=False)
    .head(10)[all_features + ['centroid_dist']]
    .rename(columns={'centroid_dist': 'Severity'})
)
display_table(priority_table, "Maintenance Priority, Top 10 Critical Anomalies by Severity")
insight("Higher severity = farther from the nearest healthy cluster centroid = more extreme fault state.")





#8. Conclusion & Final Recommendations

Markdown cell summarizing the tradeoffs (IQR for speed, SVM for density boundaries, IF for complex interactions without scaling).

State the final recommendation (Isolation Forest) based on operational tradeoffs.

In [ ]:
# @title  7.1 — Key Findings

display(HTML("""
<div style='background:#0F172A;border:1px solid #1E293B;border-radius:12px;
            padding:28px 32px;margin:24px 0;'>
  <p style='margin:0 0 12px;font-size:13px;font-weight:700;letter-spacing:.06em;
            text-transform:uppercase;color:#38BDF8;font-family:Inter,sans-serif'>
    Key Findings & Recommendations
  </p>
  <ul style='margin:0;padding-left:20px;color:#CBD5E1;font-family:Inter,sans-serif;
             font-size:13px;line-height:1.8'>
    <li><b>IQR baseline:</b> threshold=2 effectively filters single-sensor noise
        while catching genuine multi-system faults (~1.6% of readings).</li>
    <li><b>OCSVM:</b> captures multivariate anomalies invisible to IQR;
        nu=0.03 / gamma=0.5 provides the best balance.</li>
    <li><b>Isolation Forest:</b> consistent with OCSVM at contamination=0.03;
        tree-based logic complements the kernel boundary.</li>
    <li><b>Feature engineering:</b> lub_efficiency and power_intensity rank highest
        in feature importance, validating the 12-feature expansion.</li>
    <li><b>Consensus strategy:</b> IF + OCSVM agreement = most reliable anomaly set;
        triple consensus gives maximum confidence.</li>
    <li><b>Next steps:</b> deploy centroid-distance scoring for real-time severity ranking;
        monitor lub_efficiency and power_intensity as primary alert triggers.</li>
  </ul>
</div>
"""))




In [ ]:
# @title  7.2 — Best Method Selection

display(HTML("""
<div style='background:#0F172A;border:1px solid #38BDF8;border-radius:12px;
            padding:28px 32px;margin:16px 0;'>
  <p style='margin:0 0 14px;font-size:13px;font-weight:700;letter-spacing:.06em;
            text-transform:uppercase;color:#38BDF8;font-family:Inter,sans-serif'>
    Approach Comparison & Best Method Selection
  </p>
  <table style="width:100%;border-collapse:collapse;font-family:Inter,sans-serif;font-size:12px">
    <thead>
      <tr style="background:#1E293B;color:#38BDF8">
        <th style="padding:8px 12px;text-align:left">Method</th>
        <th style="padding:8px 12px">Multivariate</th>
        <th style="padding:8px 12px">Scaling</th>
        <th style="padding:8px 12px">Speed</th>
        <th style="padding:8px 12px">Interpretability</th>
        <th style="padding:8px 12px">Rate Achieved</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#111827;color:#CBD5E1">
        <td style="padding:7px 12px"><b>IQR</b></td>
        <td style="padding:7px 12px;text-align:center">No — per-sensor only</td>
        <td style="padding:7px 12px;text-align:center">Optional</td>
        <td style="padding:7px 12px;text-align:center">Very fast</td>
        <td style="padding:7px 12px;text-align:center">High</td>
        <td style="padding:7px 12px;text-align:center">~1.6% (threshold=2)</td>
      </tr>
      <tr style="background:#0F172A;color:#CBD5E1">
        <td style="padding:7px 12px"><b>One-Class SVM</b></td>
        <td style="padding:7px 12px;text-align:center">Yes — kernel space</td>
        <td style="padding:7px 12px;text-align:center">Required</td>
        <td style="padding:7px 12px;text-align:center">Slow at scale</td>
        <td style="padding:7px 12px;text-align:center">Medium</td>
        <td style="padding:7px 12px;text-align:center">~3.0% (nu=0.03)</td>
      </tr>
      <tr style="background:#111827;color:#CBD5E1">
        <td style="padding:7px 12px"><b>Isolation Forest</b></td>
        <td style="padding:7px 12px;text-align:center">Yes — tree partitions</td>
        <td style="padding:7px 12px;text-align:center">Not required</td>
        <td style="padding:7px 12px;text-align:center">Fast</td>
        <td style="padding:7px 12px;text-align:center">Medium</td>
        <td style="padding:7px 12px;text-align:center">~3.0% (contam=0.03)</td>
      </tr>
    </tbody>
  </table>
  <p style='margin:16px 0 6px;font-size:12.5px;color:#F1F5F9;font-family:Inter,sans-serif;
            font-weight:700'>Selected method: Isolation Forest</p>
  <p style='margin:0;font-size:12.5px;color:#CBD5E1;font-family:Inter,sans-serif;line-height:1.8'>
    (1) Detects complex multivariate anomalies IQR cannot.
    (2) Does not require feature scaling — simpler production pipeline.
    (3) Significantly faster than OCSVM at scale — viable for near-real-time monitoring.
    (4) Achieves the target anomaly rate with strong PCA separation confirmed by the comparison plots.
    IQR is retained as a fast-path alert for individual sensor breaches.
  </p>
</div>
"""))
